In [ ]:
# =============================================================================
# JUSTIFICACIÓN DEL UMBRAL DEL TARGET
# =============================================================================
#
# Objetivo:
# Analizar la distribución diaria del área quemada para seleccionar un umbral
# adecuado para la clasificación binaria del modelo.
#
# El análisis se realiza únicamente sobre el período 2008–2022, coincidente con
# la disponibilidad de ERA5-Land.
# =============================================================================

# Período utilizado en el modelado
ba_modelo = ba.sel(time=slice("2008-01-01", "2022-12-31"))

# Extraer valores válidos
valores = ba_modelo.values.flatten()
valores = valores[~np.isnan(valores)]

# Solo incendios
positivos = valores[valores > 0]

print("=" * 60)
print("DISTRIBUCIÓN DEL ÁREA QUEMADA DIARIA")
print("=" * 60)

print(f"Observaciones totales      : {len(valores):,}")
print(f"Observaciones con incendio : {len(positivos):,}")
print(f"Prevalencia (>0 ha)        : {len(positivos)/len(valores)*100:.3f}%")

In [ ]:
print("\nDistribución de incendios (solo observaciones positivas)")
print("-" * 60)

for p in [50, 75, 90, 95, 99]:
    print(f"P{p:02d}: {np.percentile(positivos, p):8.1f} ha")

print()
print(f"Media    : {positivos.mean():8.1f} ha")
print(f"Mediana  : {np.median(positivos):8.1f} ha")
print(f"Máximo   : {positivos.max():8.1f} ha")

In [ ]:
# =============================================================================
# DISTRIBUCIÓN LOGARÍTMICA DEL ÁREA QUEMADA
# =============================================================================

fig, ax = plt.subplots(figsize=(8,5))

sns.histplot(
    np.log1p(positivos),
    bins=60,
    ax=ax
)

ax.set_title("Distribución diaria del área quemada")
ax.set_xlabel("log(1 + área quemada [ha])")
ax.set_ylabel("Frecuencia")

plt.tight_layout()

plt.savefig(
    FIGURES / "distribucion_area_quemada.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =============================================================================
# IMPACTO DEL UMBRAL SOBRE EL DESBALANCE DE CLASES
# =============================================================================

umbrales = [0, 1, 5, 10, 25, 50, 100, 250, 500]

resultados = []

for u in umbrales:

    positivos = (ba_modelo >= u).sum().item()

    prevalencia = positivos / len(valores) * 100

    resultados.append({
        "Umbral (ha)": u,
        "Positivos": positivos,
        "Prevalencia (%)": prevalencia
    })

df_umbral = pd.DataFrame(resultados)

print(df_umbral)

In [ ]:
# =============================================================================
# PREVALENCIA SEGÚN EL UMBRAL
# =============================================================================

fig, ax = plt.subplots(figsize=(8,5))

ax.plot(
    df_umbral["Umbral (ha)"],
    df_umbral["Prevalencia (%)"],
    marker="o",
    linewidth=2
)

# Destacar el umbral seleccionado
ax.scatter(
    10,
    df_umbral.loc[df_umbral["Umbral (ha)"] == 10, "Prevalencia (%)"],
    s=120,
    color="red",
    label="Umbral seleccionado"
)

ax.legend()

ax.set_xlabel("Umbral de área quemada (ha)")
ax.set_ylabel("Prevalencia (%)")
ax.set_title("Efecto del umbral sobre la prevalencia del target")

ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    FIGURES / "seleccion_umbral_target.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()